In [1]:
from tqdm.notebook import tqdm
for i in tqdm(range(100)):
    pass

  0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
import numpy as np

In [2]:
import mlflow

In [3]:

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("gingo")

2026/03/27 23:46:47 INFO mlflow.tracking.fluent: Experiment with name 'exp1' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlruns/1', creation_time=1774655207177, experiment_id='1', last_update_time=1774655207177, lifecycle_stage='active', name='exp1', tags={}, workspace='default'>

In [3]:
import polars as pl
data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

In [4]:
data_new = data_pl.with_columns(
    pl.col("含水率").log().alias("含水率_log")
)

In [5]:
data_new.head(1)

sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4134.82018,4130.96307,4127.10596,4123.24886,4119.39175,4115.53464,4111.67753,4107.82042,4103.96331,4100.10621,4096.2491,4092.39199,4088.53488,4084.67777,4080.82066,4076.96356,4073.10645,4069.24934,4065.39223,4061.53512,4057.67801,4053.82091,4049.9638,4046.10669,4042.24958,4038.39247,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139,含水率_log
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,1,"""イチョウ""",216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.4147,0.41452,0.41427,0.41392,0.41364,0.41354,0.41355,0.41352,0.41337,0.41312,0.41284,0.4127,0.41265,0.41257,0.41242,0.4123,0.41226,0.41218,0.41193,0.41159,0.41129,0.41111,0.41102,0.41096,0.41098,0.41109,0.41116,0.41104,…,1.19969,1.20339,1.20819,1.21138,1.21194,1.21262,1.21483,1.21592,1.21666,1.22043,1.22559,1.22777,1.22751,1.22862,1.23165,1.23629,1.24109,1.24147,1.23927,1.24074,1.24282,1.24082,1.23906,1.24128,1.24471,1.24783,1.25104,1.24925,1.24145,1.2362,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219,5.375876


In [6]:
import polars as pl
import numpy as np
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ===== MLflow設定 =====
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("gingo")

# ===== データ =====
pl_df = data_new 

# 目的変数
y = pl_df["含水率_log"].to_numpy()

# 不要列除外
drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]

X = pl_df.drop(drop_cols).to_numpy()

# ===== 分割 =====
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== 学習 =====
params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 5,
    "n_jobs": -1
}

with mlflow.start_run():

    # パラメータ記録
    mlflow.log_params(params)

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)

    # 予測
    pred = model.predict(X_valid)

    # RMSE（squared=False使えない環境対応）
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    # ログ
    mlflow.log_metric("rmse", rmse)

    # モデル保存
    mlflow.lightgbm.log_model(model, "model")

    print("RMSE:", rmse)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027973 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 396525
[LightGBM] [Info] Number of data points in the train set: 1057, number of used features: 1555
[LightGBM] [Info] Start training from score 3.408297


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/03/28 00:06:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


2026/03/28 00:06:39 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.18669948455892013
🏃 View run entertaining-cub-868 at: http://mlflow:5000/#/experiments/1/runs/0b6458984058453b94948314bb0a87ed
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [12]:
!head ../data/sample_submit.csv

95,50
96,50
97,50
98,50
99,50
100,50
101,50
102,50
103,50
104,50


In [15]:
import mlflow

mlflow.set_tracking_uri("http://mlflow:5000")

# exp1のrun取得
runs = mlflow.search_runs(experiment_names=["exp1"])

# RMSE最小のrunを使う
best_run = runs.sort_values("metrics.rmse").iloc[0]
run_id = best_run["run_id"]

# モデルロード
model = mlflow.lightgbm.load_model(f"runs:/{run_id}/model")

In [18]:
!ls ../data/test.csv

test_data =  pl.read_csv(r"../data/test.csv",encoding="shift_jis")

../data/test.csv


In [20]:
test_data.head()

sample number,species number,樹種,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,9866.48323,…,4138.67729,4134.82018,4130.96307,4127.10596,4123.24886,4119.39175,4115.53464,4111.67753,4107.82042,4103.96331,4100.10621,4096.2491,4092.39199,4088.53488,4084.67777,4080.82066,4076.96356,4073.10645,4069.24934,4065.39223,4061.53512,4057.67801,4053.82091,4049.9638,4046.10669,4042.24958,4038.39247,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
95,2,"""クスノキ""",0.32877,0.32855,0.32823,0.32792,0.32759,0.32729,0.32713,0.32703,0.32688,0.32674,0.32668,0.32665,0.32654,0.32627,0.32591,0.32565,0.3256,0.32555,0.32529,0.32495,0.32474,0.32461,0.32447,0.32432,0.3241,0.32385,0.32373,0.32354,0.32318,0.32293,0.32281,0.32268,0.32257,0.32246,…,1.27477,1.27645,1.28061,1.286,1.29,1.29307,1.29534,1.29671,1.29833,1.30164,1.30746,1.31474,1.32133,1.32722,1.33457,1.34164,1.34486,1.34568,1.34588,1.34549,1.34356,1.34215,1.34756,1.35521,1.35478,1.34898,1.3474,1.35443,1.36345,1.3647,1.3611,1.35863,1.35918,1.36074,1.35224,1.33495,1.32482
96,2,"""クスノキ""",0.30061,0.30006,0.29978,0.29984,0.29977,0.29945,0.29914,0.29891,0.29878,0.29874,0.29868,0.29861,0.29856,0.29835,0.29795,0.29768,0.29773,0.29783,0.29774,0.29758,0.29743,0.29719,0.29691,0.29666,0.29644,0.29626,0.29612,0.29591,0.29561,0.29545,0.29555,0.29578,0.29592,0.2958,…,1.03124,1.03431,1.03789,1.04243,1.04693,1.05064,1.05287,1.05412,1.05658,1.06155,1.0672,1.07028,1.0707,1.07163,1.07439,1.07874,1.08518,1.09105,1.09306,1.09374,1.09617,1.10041,1.1048,1.10675,1.10801,1.11193,1.11707,1.12196,1.12506,1.12357,1.11911,1.11655,1.11955,1.12671,1.13182,1.1307,1.12377
97,2,"""クスノキ""",0.26048,0.26033,0.26,0.2597,0.25952,0.25945,0.25927,0.25895,0.2588,0.25893,0.25901,0.25881,0.25858,0.25841,0.25822,0.25812,0.25813,0.25803,0.25782,0.25769,0.25764,0.2575,0.25729,0.25715,0.25705,0.25695,0.25692,0.25686,0.25668,0.25657,0.25663,0.25683,0.25691,0.25674,…,0.89374,0.89657,0.90014,0.90319,0.90619,0.91003,0.91351,0.91544,0.91733,0.92038,0.9239,0.92724,0.93072,0.93516,0.93921,0.94099,0.94286,0.94782,0.95365,0.95759,0.96083,0.96385,0.96591,0.96823,0.97249,0.97719,0.98016,0.9837,0.98904,0.99325,0.9961,0.99886,0.99919,0.99367,0.98723,0.98868,0.99461
98,2,"""クスノキ""",0.2321,0.23188,0.23167,0.23158,0.23142,0.23122,0.23119,0.23116,0.23097,0.23086,0.2308,0.2306,0.23037,0.23023,0.23008,0.22995,0.22987,0.22979,0.22972,0.22969,0.22969,0.22956,0.2293,0.22912,0.22899,0.22884,0.22878,0.22878,0.22869,0.22861,0.22868,0.22886,0.22893,0.22875,…,0.8384,0.8422,0.84598,0.84936,0.85252,0.8557,0.85856,0.86089,0.86352,0.86693,0.87108,0.87612,0.88105,0.88431,0.88654,0.88979,0.89458,0.89964,0.90425,0.90969,0.91623,0.9204,0.92072,0.92142,0.92462,0.9281,0.932,0.9383,0.94524,0.94897,0.95018,0.95243,0.95574,0.95703,0.9553,0.95444,0.95832
99,2,"""クスノキ""",0.19725,0.19704,0.19687,0.19676,0.1966,0.19644,0.19639,0.19632,0.19623,0.1962,0.19619,0.1961,0.196,0.19591,0.19574,0.19561,0.19557,0.1955,0.19541,0.19532,0.19522,0.19512,0.19501,0.19483,0.19458,0.19444,0.19446,0.19436,0.19413,0.1941,0.19423,0.19434,0.1944,0.19439,…,0.77358,0.7768,0.77961,0.7823,0.78544,0.78958,0.79374,0.7965,0.7989,0.80301,0.80834,0.8129,0.81568,0.81723,0.82018,0.82557,0.83049,0.83409,0.83926,0.84652,0.85239,0.85508,0.85679,0.85862,0.86003,0.86343,0.87088,0.87793,0.88175,0.88647,0.89267,0.8979,0.90191,0.90292,0.90119,0.89835,0.89623


In [21]:

pl_test = test_data

drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]

X_test = pl_test.drop(drop_cols, strict=False).to_numpy()

In [29]:
y_pred_log = model.predict(X_test)

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [32]:
import pandas as pd

y_pred = np.exp(y_pred_log)

submission = pd.DataFrame({
    "id": pl_test["sample number"].to_numpy(),
    "含水率log": y_pred
})

submission.head()

,id,含水率log
0,95,155.217350
1,96,142.366065
2,97,139.324560
3,98,170.698454
4,99,158.247433


In [47]:
submission.to_csv(r"../data/submission.csv",index=False,header = False)

In [48]:
!head ../data/sample_submit.csv

95,50
96,50
97,50
98,50
99,50
100,50
101,50
102,50
103,50
104,50


In [49]:
!head ../data/submission.csv

95,155.21734950644017
96,142.36606481752068
97,139.32455958367885
98,170.69845412595623
99,158.2474328520994
100,133.35389637964118
101,121.29102172161932
102,138.55922050521977
103,162.21300927161576
104,143.97672793012214
